In [1]:
"""
Statistical Analysis Toolkit

RULE: implement every calculation yourself with plain Python.
NumPy may be imported ONLY in compare_with_numpy(), to verify your results.
Do not use the statistics module.
"""
import csv
import math
from pathlib import Path

DATA_PATH = Path("data") / "agent_deposits.csv"

---

## cleaning data

In [15]:
def load_column(path: Path, column: str) -> tuple[list[float], list[tuple[int, str, str]]]:
    """Return (valid_values, rejected) where rejected holds (row_number, raw_value) pairs."""
    valid_values, rejected = [], []

    # importing data using dict reader
    with open(path, newline="") as f:
        for index, row in enumerate(csv.DictReader(f), start=2):
            raw_data = row[column]
            try:
                # attmept to convert col to float
                valid_values.append(float(raw_data))
            except ValueError:
                # if float conversion fails, find reason 
                reason = "missing value" if raw_data.strip() == "" else "Not a number"
                rejected.append((index, raw_data, reason)) 

    return valid_values, rejected


# viewing unusable data in daily deposit and transaction cols
valid_deposits, rejected_deposits = load_column(DATA_PATH, "daily_deposits_kes")

print(rejected_deposits)

valid_txns, rejected_txns = load_column(DATA_PATH, "transactions")
print(rejected_txns)

# print(valid_deposits)
# 6 columns unuseable

[(7, '', 'missing value'), (10, '-', 'Not a number'), (14, 'n/a', 'Not a number'), (25, '12,40O', 'Not a number'), (29, '', 'missing value')]
[(7, '', 'missing value'), (10, '', 'missing value'), (14, '', 'missing value'), (17, 'abc', 'Not a number'), (29, '', 'missing value')]


---

In [3]:
def mean(values: list[float]) -> float:
    return sum(values) / len(values)


# finding mean of valid deposits
deposits_mean = mean(valid_deposits)
print("Mean of valid deposits")
print(f"{deposits_mean:,.2f}")
print()

# finding mean of valid transactions
print("Mean of vlaid txns")
txns_mean = mean(valid_txns)
print(f"{txns_mean:,.2f}")

Mean of valid deposits
22,400.00

Mean of vlaid txns
93.11


In [4]:
def _median_sorted(s: list[float]) -> float:
    mid = len(s) // 2
    return s[mid] if len(s) % 2 else (s[mid - 1] + s[mid]) / 2

def median(values: list[float]) -> float:
    return _median_sorted(sorted(values))

# finding median of valid deposits
deposits_median = median(valid_deposits)
print("Deposits median")
print(deposits_median)
print()

# finding meadian of valid txns
print("Txns median")
txns_median = median(valid_txns)
print(txns_median)


Deposits median
18805.47

Txns median
94.0


In [5]:
def mode(values: list[float]) -> list[float]:
    """Return every value that occurs most often (a list, because there can be ties or none)."""
    # initialize empty set counts to hold unique values
    counts = {}

    # count every time a value is found, start count at 0, add 1 for each find
    for value in values:
        counts[value] = counts.get(value, 0) + 1

    # if no value is found, no mode, return empty list
    if not counts:
        return []

    # find highest count
    highest = max(counts.values())
    # if all values only appear once, no mode, return empty list
    if highest == 1:
        return []

    # return value if it has the highest count
    return [value for value, count in counts.items() if count == highest]


# finding mode
# finding txns mode
txns_mode = mode(valid_txns)
print("Txns Mode")
print(txns_mode)



Txns Mode
[132.0, 73.0]


In [6]:
def variance(values: list[float], sample: bool = True) -> float:
    """Sample variance (n - 1) by default; population variance (n) when sample is False."""
    # finding count of values passed
    n = len(values)
    # if < 2 values
    if n < 2:
        raise ValueError("Need at least 2 values to calculate vaariance!")
    data_mean = mean(values)
    # find diff of value - mean
    # square diff, to make it posive
    # sum all +ve squared values
    squared_diffs = sum((v- data_mean) ** 2 for v in values)
    # divide sum by count - 1
    return squared_diffs / (n - 1 if sample else n)


# test variance
deposits_var = variance(valid_deposits)
print("Deposits Variance")
print(deposits_var)
print()

txns_var = variance(valid_txns)
print("Txns variance")
print(txns_var)

Deposits Variance
328019873.35939974

Txns variance
502.5747899159664


In [7]:
def std_dev(values: list[float], sample: bool = True) -> float:
    # std dev id square root of variance so use power 0.5 to find square root
    return variance(values, sample) ** 0.5


# test std dev
deposits_std_dev = std_dev(valid_deposits)
print("Deposits std dev")
print(deposits_std_dev)
print()

txns_std_dev = std_dev(valid_txns)
print("Txns std dev")
print(txns_std_dev)

Deposits std dev
18111.318929315992

Txns std dev
22.418179897484237


In [8]:
def _quartiles_sorted(s: list[float]) -> tuple[float, float, float]:
    if len(s) < 2:
        raise ValueError("need at least 2 values to calculate quartiles")
    half = len(s) // 2
    return _median_sorted(s[:half]), _median_sorted(s), _median_sorted(s[len(s) - half:])

def quartiles(values: list[float]) -> tuple[float, float, float]:
    return _quartiles_sorted(sorted(values))


# testing quartiles
deposits_quats = quartiles(valid_deposits)
print("deposits quartiles")
print(deposits_quats)
print()

txns_quats = quartiles(valid_txns)
print("txns quartiles")
print(txns_quats)
print()

deposits quartiles
(15988.21, 18805.47, 21195.44)

txns quartiles
(73.0, 94.0, 110.0)



In [9]:
def iqr_outliers(values: list[float], k: float = 1.5) -> dict:
    """Return the IQR, the lower and upper fences, and the values outside them."""
    q1, _, q3 = quartiles(values)
    iqr = q3 - q1
    lower_limit = q1 - k * iqr
    upper_limit = q3 + k * iqr

    return {
        "iqr" : iqr,
        "lower_limit" : lower_limit,
        "upper_limit": upper_limit,
        # return value from values
        "outliers" : [v for v in values 
                      # if value < or > than limits
                      if v < lower_limit 
                      or v > upper_limit],
    }


# testing IQR
deposits_IQR = iqr_outliers(valid_deposits)
print("deposits outliers")
print(deposits_IQR)
print()

txns_IQR = iqr_outliers(valid_txns)
print("txns quartiles")
print(txns_IQR)
print()

deposits outliers
{'iqr': 5207.23, 'lower_limit': 8177.365, 'upper_limit': 29006.284999999996, 'outliers': [96500.0, 1200.0, 88400.0, 29060.99]}

txns quartiles
{'iqr': 37.0, 'lower_limit': 17.5, 'upper_limit': 165.5, 'outliers': []}



---

In [16]:
class StatisticsAnalyzer:
    """Wraps one numeric column and exposes every statistic above as a method."""

    def __init__(self, values, name="values", rejected=None, labels=None):
        self.name = name
        self.rejected = rejected or []
        self.values = list(values)
        self.labels = labels or [None] * len(values)
        s = sorted(self.values)  # sorted once, used for everything below
        n = len(s)
        q1, median, q3 = _quartiles_sorted(s)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        var = variance(s)

        self._outliers = [(v, lab) for v, lab in zip(self.values, self.labels)
                          if v < lower or v > upper]
        
        self._summary = {
            "count": n, "mean": sum(s) / n, "median": median, "mode": mode(s),
            "variance": var, "std_dev": var ** 0.5, "min": s[0], "max": s[-1],
            "q1": q1, "q3": q3, "iqr": iqr, "lower_fence": lower, "upper_fence": upper,
            "outliers": [v for v, _ in self._outliers],
        }

    @classmethod
    def from_csv(cls, path, column):
        values, rejected = load_column(path, column)
        bad_rows = {row_number for row_number, _, _ in rejected}
        with open(path, newline="") as f:
            labels = [(row["date"], row["agent_id"])
                      for i, row in enumerate(csv.DictReader(f), start=2) if i not in bad_rows]
        return cls(values, column, rejected, labels)

    def summary(self):
        return dict(self._summary)

    def without_outliers(self):
        s = self._summary
        kept = [(v, lab) for v, lab in zip(self.values, self.labels)
                if s["lower_fence"] <= v <= s["upper_fence"]]
        return StatisticsAnalyzer([v for v, _ in kept], self.name, self.rejected,
                                  [lab for _, lab in kept])

    def report(self):
        def column(s):
            return [
                str(s["count"]),
                f"{s['mean']:,.2f}",
                f"{s['median']:,.2f}",
                ", ".join(f"{m:,.2f}" for m in s["mode"]) or "none",
                f"{s['variance']:,.2f}",
                f"{s['std_dev']:,.2f}",
                f"{s['min']:,.2f} / {s['max']:,.2f}",
                f"{s['q1']:,.2f} / {s['q3']:,.2f}",
                f"{s['iqr']:,.2f}",
            ]

        labels = ["count", "mean", "median", "mode", "variance (sample)",
                  "std dev (sample)", "min / max", "Q1 / Q3", "IQR"]
        a, b = column(self._summary), column(self.without_outliers()._summary)
        width = max(len(x) for x in a + ["all data"]) + 2

        lines = [
            f"=== Statistics report: {self.name} ===",
            f"Rows read         : {len(self.values) + len(self.rejected)}",
            f"Valid values      : {len(self.values)}",
            f"Rejected values   : {len(self.rejected)}",
        ]
        lines += [f"  row {r}: {raw!r} ({why})" for r, raw, why in self.rejected]
        lines += ["", " " * 20 + "all data".ljust(width) + "outliers removed"]
        lines += [l.ljust(20) + x.ljust(width) + y for l, x, y in zip(labels, a, b)]

        s = self._summary
        outliers = ", ".join(
            f"{v:,.2f} ({lab[0]}, {lab[1]})" if lab else f"{v:,.2f}"
            for v, lab in self._outliers
        )
        lines += [
            "",
            f"Outlier fences      : lower {s['lower_fence']:,.2f}, upper {s['upper_fence']:,.2f}",
            f"Outliers            : {outliers or 'none'}",
        ]
        return "\n".join(lines)

    

In [20]:
transactions = StatisticsAnalyzer.from_csv(DATA_PATH, "transactions")
print(transactions.report())
print()


=== Statistics report: transactions ===
Rows read         : 40
Valid values      : 35
Rejected values   : 5
  row 7: '' (missing value)
  row 10: '' (missing value)
  row 14: '' (missing value)
  row 17: 'abc' (Not a number)
  row 29: '' (missing value)

                    all data        outliers removed
count               35              35
mean                93.11           93.11
median              94.00           94.00
mode                73.00, 132.00   73.00, 132.00
variance (sample)   502.57          502.57
std dev (sample)    22.42           22.42
min / max           60.00 / 140.00  60.00 / 140.00
Q1 / Q3             73.00 / 110.00  73.00 / 110.00
IQR                 37.00           37.00

Outlier fences      : lower 17.50, upper 165.50
Outliers            : none



In [21]:
deposits = StatisticsAnalyzer.from_csv(DATA_PATH, "daily_deposits_kes")
print(deposits.report())


=== Statistics report: daily_deposits_kes ===
Rows read         : 40
Valid values      : 35
Rejected values   : 5
  row 7: '' (missing value)
  row 10: '-' (Not a number)
  row 14: 'n/a' (Not a number)
  row 25: '12,40O' (Not a number)
  row 29: '' (missing value)

                    all data               outliers removed
count               35                     31
mean                22,400.00              18,349.65
median              18,805.47              18,506.72
mode                none                   none
variance (sample)   328,019,873.36         10,116,378.52
std dev (sample)    18,111.32              3,180.63
min / max           1,200.00 / 96,500.00   12,772.70 / 26,725.69
Q1 / Q3             15,988.21 / 21,195.44  15,988.21 / 20,745.52
IQR                 5,207.23               4,757.31

Outlier fences      : lower 8,177.36, upper 29,006.28
Outliers            : 96,500.00 (2026-09-20, AG-104), 1,200.00 (2026-10-02, AG-104), 88,400.00 (2026-10-06, AG-104), 29,060.99 (

---

In [14]:


if __name__ == "__main__":
    # TODO
    pass
